# GP1 — nonlinear-regime (LAOS) training, Table 4

Identifies the coefficients $c_0 \dots c_7$ of eq. (7),

$$\dot\gamma = c_0 + c_1\sigma + c_2\dot\sigma + c_3\sigma^2 + c_4\sigma\dot\sigma
+ c_5\dot\sigma\sigma^2 + c_6\sigma^3 + c_7\dot\sigma\sigma^3 ,$$

from the stress-controlled LAOS data of GP1 ($\sigma_0$ = 10 Pa, $\omega$ = 1 rad/s).
Each sample age is trained independently: a RhINN maps $(t, \sigma_0) \mapsto \gamma$,
automatic differentiation gives $\dot\gamma$, and STRidge selects the active terms inside
the ADO loop (Section 2.2).

Two term libraries were used. The **full** library keeps all eight terms of eq. (7). At the
ages where the full fit gave an unbounded start-up shear response (1–5 h and 9 h), the data
were re-fitted with a **reduced** library without the cubic-stress terms. Table 4 reports
the reduced fit at those ages and the full fit otherwise; the `library` column below shows
which.

All training settings are in `rhinn_sindy/settings.py`.

* **Output:** `results/retrained/GP1_LAOS_full.csv` and `GP1_LAOS_reduced.csv`, written
  after every age. Table 4 itself (`results/coefficients/GP1_LAOS.csv`) is never
  overwritten.
* **Runtime:** about 40 min per age on a laptop CPU.
* **Hyperparameters:** the settings in `rhinn_sindy/settings.py` are the ones used for the
  paper. Moving from one sample age to another changes the stress and strain magnitudes by
  orders of magnitude, so an age (or a new material) may need some hyperparameter tuning,
  mainly the loss weights, learning rate, number of rounds and STRidge tolerance.

Even with the original settings, retrained coefficients differ from run to run (random
initialisation and collocation points) and can differ noticeably from Table 4.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from rhinn_sindy import figures
from rhinn_sindy.reproduce import available_ages, compare_with_paper, run_laos

MATERIAL = "GP1"
tf.random.set_seed(1234)
np.random.seed(1234)

## Choose the ages

Ages available for training (as labelled in Table 4) and the library the paper
reports at each:

In [ ]:
available_ages(MATERIAL, "LAOS")

In [ ]:
# "all" for the whole of Table 4, or a list of ages in hours, e.g. [1, 9].
AGES = "all"
# "paper" uses the library in the table above; "full" or "reduced" forces one.
LIBRARY = "paper"

## Training data

In [ ]:
figures.laos_training_data(MATERIAL)
plt.show()

## Train

In [ ]:
results = run_laos(MATERIAL, AGES, LIBRARY)

## Compare with the fits behind Table 4

In [ ]:
compare_with_paper(MATERIAL, "LAOS")

## Check: integrate eq. (7) with the trained coefficients

The stress used in training (nominal or fitted sinusoid) drives eq. (7) from the first
measured strain (up to four of the trained ages).

In [ ]:
figures.laos_fit(MATERIAL, results)
plt.show()